Library imports 

In [47]:
import pandas as pd
import s3fs
import xarray as xr
import mlflow


Connecting to Public AWS 3 Bucket and selecting required data

In [48]:
fs = s3fs.S3FileSystem(anon=True)

SYN1DEG_DAILY = (
    "s3://nasa-power/syn1deg/temporal/"
    "power_syn1deg_daily_temporal_utc.zarr"
)

MERRA2_DAILY = (
    "s3://nasa-power/merra2/temporal/"
    "power_merra2_daily_temporal_utc.zarr"
)

syn_ds = xr.open_zarr(SYN1DEG_DAILY, storage_options={"anon": True})
merra_ds = xr.open_zarr(MERRA2_DAILY, storage_options={"anon": True})


Variable Listing in syn1deg and merra2

In [49]:
syn1deg_variables = list(syn_ds.data_vars)
merra2_variables = list(merra_ds.data_vars)

print("SYN1DEG Variables:")
print(syn1deg_variables)

print("\nMERRA2 Variables:")
print(merra2_variables)

SYN1DEG Variables:
['AIRMASS', 'ALLSKY_KT', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_UP', 'ALLSKY_SFC_PAR_DIFF', 'ALLSKY_SFC_PAR_DIRH', 'ALLSKY_SFC_PAR_TOT', 'ALLSKY_SFC_SW_DIFF', 'ALLSKY_SFC_SW_DIRH', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_UP', 'ALLSKY_SFC_UV_INDEX', 'ALLSKY_SFC_UVA', 'ALLSKY_SFC_UVB', 'ALLSKY_SRF_ALB', 'AOD_55', 'AOD_55_ADJ', 'AOD_84', 'CLOUD_AMT', 'CLOUD_AMT_DAY', 'CLOUD_AMT_NIGHT', 'CLOUD_OD', 'CLRSKY_DAYS', 'CLRSKY_KT', 'CLRSKY_NKT', 'CLRSKY_SFC_LW_DWN', 'CLRSKY_SFC_LW_UP', 'CLRSKY_SFC_PAR_DIFF', 'CLRSKY_SFC_PAR_DIRH', 'CLRSKY_SFC_PAR_TOT', 'CLRSKY_SFC_SW_DIFF', 'CLRSKY_SFC_SW_DIRH', 'CLRSKY_SFC_SW_DNI', 'CLRSKY_SFC_SW_DWN', 'CLRSKY_SFC_SW_UP', 'CLRSKY_SRF_ALB', 'MIDDAY_INSOL', 'ORIGINAL_ALLSKY_SFC_SW_DIFF', 'ORIGINAL_ALLSKY_SFC_SW_DIRH', 'PSH', 'PW', 'SRF_ALB_ADJ', 'TOA_SW_DNI', 'TOA_SW_DWN', 'TS_ADJ']

MERRA2 Variables:
['CDD0', 'CDD10', 'CDD18_3', 'DISPH', 'EVLAND', 'EVPTRNS', 'FROST_DAYS', 'FRSEAICE', 'FRSNO', 'GWETPROF', 'GWETROOT',

Dimension listing from syn1deg and merra 2

In [57]:
syn1deg_dimensions = list(syn_ds.dims)
merra_dimensions = list(merra_ds.dims)

print("SYN1DEG Dimensions:")
print(syn1deg_dimensions)   

print("\nMERRA2 Dimensions:")       
print(merra_dimensions)


SYN1DEG Dimensions:
['time', 'lat', 'lon']

MERRA2 Dimensions:
['time', 'lat', 'lon']


Location filtering

In [51]:
COUNTRY_BOUNDS = {
    "Peninsular Malaysia": {
        "lat_min": 1,
        "lat_max": 8,
        "lon_min": 98,
        "lon_max": 105
    },
    "East Malaysia": {
        "lat_min" : 1, 
        "lat_max" : 7,
        "lon_min" : 108,
        "lon_max" : 120
    },
    "USA": {
        "lat_min": 24.5,
        "lat_max": 49.5,
        "lon_min": -125.0,
        "lon_max": -66.9
    }
}

def select_region(ds, bounds):
    return ds.sel(
        lat=slice(bounds["lat_min"], bounds["lat_max"]),
        lon=slice(bounds["lon_min"], bounds["lon_max"])
    )


Variable and dimension selection from syn1deg 

In [52]:
SYN1DEG_VARS = [
    "ALLSKY_SFC_SW_DWN",   # GHI (TARGET)
    "CLOUD_AMT",
    "AOD_55",
    "PW",
    "ALLSKY_KT",
    "PSH"
]


data extraction and aggregation from syn1deg for example 

In [53]:
region_data = {}

for region, bounds in COUNTRY_BOUNDS.items():
    region_data[region] = {}
    
    for var in SYN1DEG_VARS:
        da_region = select_region(syn_ds[var], bounds)
        da_daily = da_region.mean(dim=["lat", "lon"])
        region_data[region][var] = da_daily.to_series()

region_dfs = {}

for region, vars_dict in region_data.items():
    df = pd.DataFrame(vars_dict)
    df.index.name = "time"
    df = df.dropna()
    region_dfs[region] = df

region_dfs["Peninsular Malaysia"].head()


,ALLSKY_SFC_SW_DWN,CLOUD_AMT,AOD_55,PW,ALLSKY_KT,PSH
time,,,,,,
2000-12-31,111.924881,92.492645,0.322048,5.584084,0.287145,2.686333
2001-01-01,172.543488,90.471001,0.302859,5.429185,0.443673,4.140611
2001-01-02,149.142242,92.644081,0.380615,5.275720,0.384493,3.579591
2001-01-03,164.982239,92.725105,0.441635,5.185301,0.424081,3.959594
2001-01-04,136.036316,90.428154,0.462861,5.207554,0.348162,3.265097


Variable selection from merra2

In [54]:
MERRA2_VARS = [
    "RH2M",      # Relative Humidity at 2 m
    "WS2M",      # Wind Speed at 2 m
    "T2M_MAX",   # Maximum Temperature at 2 m
    "PS"         # Surface Pressure
]


data extraction and aggregation for example from merra 2

In [55]:
merra_region_data = {}

for region, bounds in COUNTRY_BOUNDS.items():
    merra_region_data[region] = {}
    
    for var in MERRA2_VARS:
        da_region = select_region(merra_ds[var], bounds)
        da_daily = da_region.mean(dim=["lat", "lon"])
        merra_region_data[region][var] = da_daily.to_series()

merra_region_dfs = {}

for region, vars_dict in merra_region_data.items():
    df = pd.DataFrame(vars_dict)
    df.index.name = "time"
    df = df.dropna()
    merra_region_dfs[region] = df

merra_region_dfs["Peninsular Malaysia"].head()


,RH2M,WS2M,T2M_MAX,PS
time,,,,
1980-12-31,84.252441,2.571834,26.815056,999.509705
1981-01-01,84.104218,2.592499,27.082165,998.377625
1981-01-02,83.419556,2.603778,27.184277,998.660400
1981-01-03,83.846672,2.715111,27.098227,999.164734
1981-01-04,83.897942,2.535948,27.009001,999.951050
